# Model cost: parameters, FLOPs and inference time

Measures the exact model used in the main notebook (same code, random weights - no dataset needed).
Settings: GPU on, Internet on (only for `pip install`). Run all cells (about 1-2 minutes) and paste the output.

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp
import math, inspect, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NC = 4
print("device:", dev, torch.cuda.get_device_name(0) if dev.type == "cuda" else "", "| torch", torch.__version__, "| smp", smp.__version__)

In [ ]:
# ---------------- FinalNet ----------------
class FinalNet(nn.Module):
    def __init__(self, num_classes, pretrained=True, enc="tu-efficientnet_b0"):
        super().__init__()
        dch = (256, 128, 64, 32, 16)
        base = smp.Unet(enc, encoder_weights=True if pretrained else None, in_channels=3, classes=1,
                        decoder_channels=dch, decoder_attention_type="scse")
        self.encoder, self.decoder, self.seg_head = base.encoder, base.decoder, base.segmentation_head
        self.aux_ids = (1, 2, 3)                                   # 3 auxiliary 1x1 heads
        self.aux_heads = nn.ModuleList([nn.Conv2d(dch[i], 1, 1) for i in self.aux_ids])
        self._f = {}
        for i in self.aux_ids:
            self.decoder.blocks[i].register_forward_hook(self._mk(i))
        ps = inspect.signature(self.decoder.forward).parameters
        self._star = any(p.kind == p.VAR_POSITIONAL for p in ps.values())   # smp-version compatibility
        c32, c16 = self.encoder.out_channels[-1], self.encoder.out_channels[-2]
        self.head = nn.Sequential(nn.Linear(c32 * 2 + c16, 256), nn.ReLU(True), nn.Dropout(0.3),
                                  nn.Linear(256, num_classes))

    def _mk(self, i):
        def hook(mod, inp, out): self._f[i] = out
        return hook

    def forward(self, x):
        feats = self.encoder(x)
        d = self.decoder(*feats) if self._star else self.decoder(feats)
        seg = self.seg_head(d)
        aux = [h(self._f[i]) for h, i in zip(self.aux_heads, self.aux_ids)]
        m = torch.sigmoid(seg.float())                              # soft mask: no threshold, no detach
        f32, f16 = feats[-1].float(), feats[-2].float()             # stride 32 and stride 16
        def wpool(f):
            mm = F.interpolate(m, size=f.shape[-2:], mode="bilinear", align_corners=False)
            return (f * mm).sum((2, 3)) / (mm.sum((2, 3)) + 1e-6)
        z = torch.cat([f32.mean((2, 3)), wpool(f32), wpool(f16)], 1)
        return dict(seg=seg, aux=aux, cls=self.head(z))

# ---------------- losses ----------------
def tversky(logit, t, alpha=0.7, eps=1.0):
    p = torch.sigmoid(logit.float())
    tp = (p * t).sum(); fn = ((1 - p) * t).sum(); fp = (p * (1 - t)).sum()
    return 1 - (tp + eps) / (tp + alpha * fn + (1 - alpha) * fp + eps)

def seg_loss(out, tgt, valid):
    if valid.sum() == 0: return None
    tot = 0; outs = [out["seg"]] + out["aux"]
    for o in outs:
        o, t = o[valid], tgt[valid]
        if o.shape[-2:] != t.shape[-2:]:
            o = F.interpolate(o.float(), size=t.shape[-2:], mode="bilinear", align_corners=False)
        tot = tot + tversky(o, t) + F.binary_cross_entropy_with_logits(o.float(), t)
    return tot / len(outs)

class Uncertainty(nn.Module):                                      # learned task weights (Kendall et al.)
    def __init__(s): super().__init__(); s.s = nn.Parameter(torch.zeros(2))
    def forward(s, ls, lc):
        t = torch.exp(-s.s[1]) * lc + s.s[1]
        if ls is not None: t = t + torch.exp(-s.s[0]) * ls + s.s[0]
        return t

class EMA:
    def __init__(s, model, decay):
        s.m = FinalNet(NC, pretrained=False).to(dev); s.m.load_state_dict(model.state_dict()); s.m.eval()
        for p in s.m.parameters(): p.requires_grad_(False)
        s.d, s.n = decay, 0
    @torch.no_grad()
    def update(s, model):
        s.n += 1; d = min(s.d, (1 + s.n) / (10 + s.n))
        for e, p in zip(s.m.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point: e.mul_(d).add_(p.detach(), alpha=1 - d)
            else: e.copy_(p)

In [ ]:
# ---------------- parameters ----------------
def n_params(m): return sum(p.numel() for p in m.parameters())
for nc in (4, 5):
    m = FinalNet(nc, pretrained=False)
    print(f"classes={nc}: total parameters = {n_params(m):,}  ({n_params(m)/1e6:.2f} M)")
m = FinalNet(4, pretrained=False)
parts = dict(encoder=m.encoder, decoder=m.decoder, seg_head=m.seg_head, aux_heads=m.aux_heads, classifier_head=m.head)
for k, v in parts.items(): print(f"  {k:16s} {n_params(v):>10,}")

In [ ]:
# ---------------- FLOPs (single forward pass, batch 1) ----------------
from torch.utils.flop_counter import FlopCounterMode
model = FinalNet(4, pretrained=False).to(dev).eval()
for size in (224, 512):
    x = torch.randn(1, 3, size, size, device=dev)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        model(x)
    fl = fc.get_total_flops()
    print(f"input 3x{size}x{size}: {fl/1e9:.2f} GFLOPs (2 x MACs)  =  {fl/2e9:.2f} GMACs")

In [ ]:
# ---------------- inference time (batch 1, 10 warm-up + 100 timed passes, mean per image) ----------------
@torch.no_grad()
def bench(size, amp, warm=10, n=100):
    x = torch.randn(1, 3, size, size, device=dev)
    for _ in range(warm):
        with torch.autocast(dev.type, enabled=amp and dev.type == "cuda"): model(x)
    if dev.type == "cuda": torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(n):
        with torch.autocast(dev.type, enabled=amp and dev.type == "cuda"): model(x)
    if dev.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000
for size in (224, 512):
    for amp in (False, True):
        print(f"3x{size}x{size} | {'fp16 autocast' if amp else 'fp32'}: {bench(size, amp):.2f} ms / image")